# Neural Operator reproduction: Darcy on GPU

Runs LNO, GNO, MGNO on the Darcy problem, using Colab GPU.


## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Clone the repo and install dependencies


In [ ]:
import os
if not os.path.isdir('neural-operator-reproduction'):
    !git clone https://github.com/gdimitrovdev/neural-operator-reproduction.git
%cd neural-operator-reproduction
!pip install -q scipy pyyaml -U gdown

## 3. Download the Darcy data


In [ ]:
import os
os.makedirs('data', exist_ok=True)
need = [f'data/piececonst_r421_N1024_smooth{i}.mat' for i in (1, 2)]
if not all(os.path.exists(f) for f in need):
    # Darcy_421.zip
    !gdown 1Z1uxG9R8AdAGJprG5STcphysjm56_0Jf -O data/Darcy_421.zip
    !cd data && unzip -o Darcy_421.zip && rm Darcy_421.zip
!ls -la data/

## 4. Train + evaluate each model

Each model trains at s=85 (500 epochs) then is evaluated at the same resolution.


In [ ]:
import subprocess, glob, os, csv, yaml

def latest_run(exp_name):
    dirs = sorted(glob.glob(f'runs/{exp_name}_*'))
    return dirs[-1] if dirs else None

def run_experiment(config):
    exp = yaml.safe_load(open(config))['logging']['experiment_name']
    print(f'Training {exp}, {config}', flush=True)
    subprocess.run(['python', 'scripts/train_darcy_operator.py', '--config', config], check=True)
    run_dir = latest_run(exp)
    print(f'Evaluating {exp}', flush=True)
    subprocess.run(['python', 'scripts/evaluate.py', '--run', run_dir, '--mode', 'same'], check=True)
    with open(os.path.join(run_dir, 'same_resolution_eval.csv')) as f:
        row = next(csv.DictReader(f))
    return run_dir, float(row['test_rel_l2']), int(row['resolution'])

results = {}
for name, cfg in [('LNO', 'configs/darcy_lno2d.yaml'),
                  ('GNO', 'configs/darcy_gno2d.yaml'),
                  ('MGNO', 'configs/darcy_mgno2d.yaml')]:
    run_dir, err, res = run_experiment(cfg)
    results[name] = (err, res, run_dir)
    print(f'{name}: rel L2 = {err:.4f} at s={res}')

## 5. Table comparison (s=85)

Our numbers vs. the paper. FNO row is filled from the local reproduction.

In [ ]:
paper = {'GNO': 0.0346, 'LNO': 0.0520, 'MGNO': 0.0416, 'FNO': 0.0108}
ours = {name: results[name][0] for name in results}
ours['FNO'] = 0.0088

print(f"{'model':6} {'ours':>10} {'paper':>10}")
for m in ['GNO', 'LNO', 'MGNO', 'FNO']:
    o = f'{ours[m]:.4f}' if m in ours else '—'
    print(f'{m:6} {o:>10} {paper[m]:>10.4f}')

## 6. Save the results

Copies the per-run artifacts into `results/` and zips them for download.

In [ ]:
import shutil, os
for name, (_, _, run_dir) in results.items():
    dest = f'results/darcy_{name.lower()}2d_s85'
    os.makedirs(dest, exist_ok=True)
    for fn in ['metrics.csv', 'same_resolution_eval.csv', 'summary.json', 'config.yaml']:
        src = os.path.join(run_dir, fn)
        if os.path.exists(src):
            shutil.copy(src, dest)
shutil.make_archive('darcy_gpu_results', 'zip', 'results')
print('Wrote results')
try:
    from google.colab import files
    files.download('darcy_gpu_results.zip')
except Exception as e:
    print('Exception: ', e)